In [2]:
%run official_eval_hotpotqa.py --gold ../data/validation.jsonl --pred predictions_full_pipeline_reordered.jsonl

[Thu Nov 27 01:44:34 2025] Read 1500 from ../data/validation.jsonl
[Thu Nov 27 01:44:34 2025] Read 1500 from predictions_full_pipeline_reordered.jsonl
{
  "em": 0.6213333333333333,
  "f1": 0.7480557687009691,
  "prec": 0.7725845016581858,
  "recall": 0.7606643171643173,
  "sp_em": 0.0,
  "sp_f1": 0.47356507936507564,
  "sp_prec": 0.33237777777776945,
  "sp_recall": 0.8273333333333334,
  "joint_em": 0.0,
  "joint_f1": 0.36529231796666417,
  "joint_prec": 0.2633757583352279,
  "joint_recall": 0.6443694758944757
}


In [3]:
%run official_eval_retrieval.py --gold ../data/validation.jsonl --pred predictions_full_pipeline_reordered.jsonl

[Thu Nov 27 01:44:47 2025] Read 1500 from ../data/validation.jsonl
[Thu Nov 27 01:44:47 2025] Read 1500 from predictions_full_pipeline_reordered.jsonl
{
  "map_at_2": 0.6193333333333333,
  "map_at_5": 0.7095055555555555,
  "map_at_10": 0.72188082010582,
  "ndcg_at_2": 0.6757048700149353,
  "ndcg_at_5": 0.7745762389600916,
  "ndcg_at_10": 0.7940925396875408,
  "recall_at_2": 0.653,
  "recall_at_5": 0.8273333333333334,
  "recall_at_10": 0.8756666666666667,
  "precision_at_2": 0.653,
  "precision_at_5": 0.3309333333333333,
  "precision_at_10": 0.17513333333333334
}


In [19]:
import json
from pathlib import Path

data_path = Path().resolve().parents[0] / "data" / "collection.jsonl"

with data_path.open("r", encoding="utf-8") as f:
    lines = f.readlines()

print(f'Total lines: {len(lines)}')

unique_texts = set()
for line in lines:
    obj = json.loads(line)
    unique_texts.add(obj["text"])

print(f'Unique texts: {len(unique_texts)}')


Total lines: 144718
Unique texts: 121178


In [23]:
import os
import sys
import json
import re
import string
from collections import Counter
from difflib import SequenceMatcher
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[0]
PRED_PATH = os.path.join(PROJECT_ROOT, 'evaluation', 'predictions_full_pipeline_reordered.jsonl')
GT_PATH = os.path.join(PROJECT_ROOT, 'data', 'validation.jsonl')

def normalize_answer(s: str) -> str:
    if s is None:
        return ""
    s = s.lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = " ".join(s.split())
    return s


def load_jsonl(path):
    items = []
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            items.append(json.loads(line))
    return items


def is_yes_no(s: str) -> bool:
    a = normalize_answer(s)
    return a in {"yes", "no"}


def numbers_only(s: str) -> str:
    return re.sub(r"[^0-9]", "", s or "")


def analyze(pred_path=PRED_PATH, gt_path=GT_PATH):
    gt_map = {o["id"]: o for o in load_jsonl(gt_path)}
    preds = load_jsonl(pred_path)

    counts = Counter()
    examples = {k: [] for k in [
        "match_exact",
        "yes_no_normalized",
        "pred_superset",
        "pred_subset",
        "numeric_format",
        "prefix_suffix_noise",
        "high_similarity",
        "other_mismatch",
    ]}

    for p in preds:
        qid = p.get("id")
        pred_ans = (p.get("answer") or "").strip()
        if qid not in gt_map:
            continue
        gt_ans = (gt_map[qid].get("answer") or "").strip()

        pn = normalize_answer(pred_ans)
        gn = normalize_answer(gt_ans)

        if pn == gn:
            counts["match_exact"] += 1
            if len(examples["match_exact"]) < 5:
                examples["match_exact"].append((pred_ans, gt_ans))
            continue

        # yes/no normalize catch (if raw has extra text around yes/no)
        if is_yes_no(pred_ans) and is_yes_no(gt_ans) and normalize_answer(pred_ans) == normalize_answer(gt_ans):
            counts["yes_no_normalized"] += 1
            if len(examples["yes_no_normalized"]) < 5:
                examples["yes_no_normalized"].append((pred_ans, gt_ans))
            continue

        # prefix/suffix noise (e.g., observed answer:, quotes, trailing dots)
        if re.search(r"^(observed\s*answer|obvious\s*answer|answer)\s*:\s*", pred_ans, re.IGNORECASE):
            stripped = re.sub(r"^(observed\s*answer|obvious\s*answer|answer)\s*:\s*", "", pred_ans, flags=re.IGNORECASE).strip().strip('"').strip("'")
            if normalize_answer(stripped) == gn:
                counts["prefix_suffix_noise"] += 1
                if len(examples["prefix_suffix_noise"]) < 5:
                    examples["prefix_suffix_noise"].append((pred_ans, gt_ans))
                continue

        # substring superset/subset
        if pn and gn and pn.find(gn) != -1 and pn != gn:
            counts["pred_superset"] += 1
            if len(examples["pred_superset"]) < 5:
                examples["pred_superset"].append((pred_ans, gt_ans))
            continue
        if pn and gn and gn.find(pn) != -1 and pn != gn:
            counts["pred_subset"] += 1
            if len(examples["pred_subset"]) < 5:
                examples["pred_subset"].append((pred_ans, gt_ans))
            continue

        # numeric formatting differences ($, %, commas)
        if numbers_only(pred_ans) and numbers_only(gt_ans) and numbers_only(pred_ans) == numbers_only(gt_ans):
            counts["numeric_format"] += 1
            if len(examples["numeric_format"]) < 5:
                examples["numeric_format"].append((pred_ans, gt_ans))
            continue

        # high similarity by ratio (likely alias/extra tokens)
        sim = SequenceMatcher(None, pn, gn).ratio()
        if sim >= 0.8:
            counts["high_similarity"] += 1
            if len(examples["high_similarity"]) < 5:
                examples["high_similarity"].append((pred_ans, gt_ans))
            continue

        counts["other_mismatch"] += 1
        if len(examples["other_mismatch"]) < 5:
            examples["other_mismatch"].append((pred_ans, gt_ans))

    total = len(preds)
    print("--- Mismatch Analysis ---")
    for k in [
        "match_exact",
        "yes_no_normalized",
        "prefix_suffix_noise",
        "pred_superset",
        "pred_subset",
        "numeric_format",
        "high_similarity",
        "other_mismatch",
    ]:
        v = counts.get(k, 0)
        pct = (v / total * 100.0) if total else 0.0
        print(f"{k}: {v} ({pct:.1f}%)")

    # Optional: show a few examples per category
    for k, exs in examples.items():
        if not exs:
            continue
        print(f"\n{k} examples:")
        for a, b in exs:
            print(f"pred='{a}' | gt='{b}'")


if __name__ == "__main__":
    analyze()




--- Mismatch Analysis ---
match_exact: 932 (62.1%)
yes_no_normalized: 0 (0.0%)
prefix_suffix_noise: 0 (0.0%)
pred_superset: 109 (7.3%)
pred_subset: 102 (6.8%)
numeric_format: 11 (0.7%)
high_similarity: 27 (1.8%)
other_mismatch: 319 (21.3%)

match_exact examples:
pred='Brawn GP' | gt='Brawn GP'
pred='Grant Imahara' | gt='Grant Imahara'
pred='Terry Southern' | gt='Terry Southern'
pred='Rihanna' | gt='Rihanna'
pred='Meadowbank Gold Mine' | gt='Meadowbank Gold Mine'

pred_superset examples:
pred='Republican Party' | gt='Republican'
pred='The song samples Nipsey Russell's song "What Would I Do If I Could Feel" from "The Wiz.' | gt='The Wiz'
pred='Magazines' | gt='magazine'
pred='Cadillac Fairview (CF)' | gt='Cadillac Fairview'
pred='United States Supreme Court' | gt='The Supreme Court'

pred_subset examples:
pred='Kirkham' | gt='Kirkham in Lancashire'
pred='50%' | gt='50 percent'
pred='Kersal' | gt='the Kersal area'
pred='Thredbo landslide' | gt='1997 Thredbo landslide'
pred='Jonesboro' | g